# 06 — Models M0 → M2

**Day 4.** Fit the empty 3-level model (M0), add L1 fixed effects (M1), then add L2 macro controls (M2). Report variance components, ICC, L2 VPC, and Snijders-Bosker proportional reduction across the three.

## Plan §6 mapping

| Model | Specification |
| --- | --- |
| M0 | Empty 3-level: `trust ~ 1 + (1\|cntry) + (1\|cy:cntry)` |
| M1 | M0 + L1 fixed effects: education, age, gender, urban/rural, main activity, income |
| M2 | M1 + L2 macro: gdp_growth, unemp_rate, hicp_inflation + round dummies |

In [1]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT.name != "MLA" and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.mla.models import (  # noqa: E402
    add_country_year_key,
    build_trust_composite,
    fit_3level,
    master_table,
    proportional_variance_reduction,
    variance_components,
)

ANALYSIS_DIR = REPO_ROOT / "data" / "analysis"
INTERIM_DIR = REPO_ROOT / "data" / "interim"
REPO_ROOT

PosixPath('/Users/karlalucic/Code/coursework/KUL/2sem/MLA')

## 1. Load + prep the analysis frame

Recode sentinel codes (66, 77, 88, 99, 555, 666, 777, 888, 999) to NaN before modelling. Build the trust composite and the country-year key.

In [2]:
def recode_sentinels(s: pd.Series, lo: float, hi: float) -> pd.Series:
    return s.where(s.between(lo, hi))

raw = pd.read_parquet(ANALYSIS_DIR / "analysis.parquet")
df = raw.copy()

# Outcome composite (item-z standardised, then averaged).
df = build_trust_composite(df)
df = add_country_year_key(df)

# L1 controls — coerce sentinels to NaN per item's valid range.
df["agea"]    = recode_sentinels(df["agea"], 14, 110)
df["gndr"]    = recode_sentinels(df["gndr"], 1, 2)
df["eisced"]  = recode_sentinels(df["eisced"], 0, 7)
df["hinctnta"] = recode_sentinels(df["hinctnta"], 1, 10)
df["mnactic"] = recode_sentinels(df["mnactic"], 1, 9)
df["domicil"] = recode_sentinels(df["domicil"], 1, 5)

# Centred age (at 45) and quadratic.
df["agea_c"] = df["agea"] - 45
df["agea_c_sq"] = df["agea_c"] ** 2

# Categorical encoders.
df["female"]  = (df["gndr"] == 2).astype("float64")
df["eisced"]  = df["eisced"].astype("float64")
df["mnactic"] = df["mnactic"].astype("float64")
df["domicil"] = df["domicil"].astype("float64")

df.shape
# statsmodels patsy formulas don't accept pandas nullable Int dtype
# (essround / isco08 / year). Coerce to plain numpy float64.
for _c in ("essround", "isco08", "year"):
    if str(df[_c].dtype).startswith("Int"):
        df[_c] = df[_c].astype("float64")


## 2. M0 — empty 3-level

In [3]:
df_m0 = df[df["trust"].notna()].copy()
print(f"M0 sample: {len(df_m0):,} obs across {df_m0['cntry'].nunique()} countries × {df_m0['country_year'].nunique()} country-years")
res0 = fit_3level("trust ~ 1", df_m0)
vc0 = variance_components(res0)
print()
print(f"sigma_u0_sq (L3 country):       {vc0.sigma_u0_sq:.4f}")
print(f"sigma_v0_sq (L2 country-year):  {vc0.sigma_v0_sq:.4f}")
print(f"sigma_e_sq  (L1 individual):    {vc0.sigma_e_sq:.4f}")
print(f"ICC L3:  {vc0.icc_l3:.3f}   (plan §2 expects 0.15–0.25)")
print(f"VPC L2:  {vc0.vpc_l2:.3f}   (plan §2 expects 0.03–0.08; ≥0.02 is the soft floor)")
print(f"VPC L1:  {vc0.vpc_l1:.3f}   (plan §2 expects 0.70–0.85)")

M0 sample: 275,254 obs across 36 countries × 154 country-years



sigma_u0_sq (L3 country):       0.1910
sigma_v0_sq (L2 country-year):  0.0174
sigma_e_sq  (L1 individual):    0.5689
ICC L3:  0.246   (plan §2 expects 0.15–0.25)
VPC L2:  0.022   (plan §2 expects 0.03–0.08; ≥0.02 is the soft floor)
VPC L1:  0.732   (plan §2 expects 0.70–0.85)


## 3. M1 — add L1 fixed effects

In [4]:
M1_FORMULA = (
    "trust ~ genai_i + C(eisced) + agea_c + agea_c_sq + female "
    "+ C(mnactic) + C(domicil) + hinctnta"
)
df_m1 = df.dropna(subset=["trust", "genai_i", "eisced", "agea_c", "female",
                          "mnactic", "domicil", "hinctnta"]).copy()
print(f"M1 sample: {len(df_m1):,} obs")
res1 = fit_3level(M1_FORMULA, df_m1)
vc1 = variance_components(res1)
print(f"ICC L3 after L1 controls: {vc1.icc_l3:.3f}")
print(f"VPC L2 after L1 controls: {vc1.vpc_l2:.3f}")
pr01 = proportional_variance_reduction(res0, res1)
print()
print("Snijders-Bosker proportional variance reduction M0 → M1:")
for k, v in pr01.items():
    print(f"  {k}: {v:+.1f}%")

M1 sample: 187,458 obs


ICC L3 after L1 controls: 0.256
VPC L2 after L1 controls: 0.020

Snijders-Bosker proportional variance reduction M0 → M1:
  delta_sigma_u0_sq_pct: +3.5%
  delta_sigma_v0_sq_pct: +16.7%
  delta_sigma_e_sq_pct: +8.2%


## 4. M2 — add L2 macro controls + round dummies

In [5]:
M2_FORMULA = (
    M1_FORMULA + " + gdp_growth + unemp_rate + hicp_inflation + C(essround)"
)
df_m2 = df.dropna(
    subset=[
        "trust", "genai_i", "eisced", "agea_c", "female", "mnactic", "domicil",
        "hinctnta", "gdp_growth", "unemp_rate", "hicp_inflation",
    ]
).copy()
print(f"M2 sample: {len(df_m2):,} obs ({df_m2['cntry'].nunique()} countries)")
res2 = fit_3level(M2_FORMULA, df_m2)
vc2 = variance_components(res2)
print(f"ICC L3 after L2 macro: {vc2.icc_l3:.3f}")
print(f"VPC L2 after L2 macro: {vc2.vpc_l2:.3f}")
print()
print("Variance reduction M0 → M2 (the cumulative effect of all controls):")
for k, v in proportional_variance_reduction(res0, res2).items():
    print(f"  {k}: {v:+.1f}%")

M2 sample: 165,969 obs (30 countries)


/Users/karlalucic/Code/coursework/KUL/2sem/MLA/.venv/lib/python3.12/site-packages/numpy/linalg/_linalg.py:2349: RuntimeWarning: divide by zero encountered in slogdet
  sign, logdet = _umath_linalg.slogdet(a, signature=signature)
/Users/karlalucic/Code/coursework/KUL/2sem/MLA/.venv/lib/python3.12/site-packages/numpy/linalg/_linalg.py:2349: RuntimeWarning: overflow encountered in slogdet
  sign, logdet = _umath_linalg.slogdet(a, signature=signature)
/Users/karlalucic/Code/coursework/KUL/2sem/MLA/.venv/lib/python3.12/site-packages/numpy/linalg/_linalg.py:2349: RuntimeWarning: invalid value encountered in slogdet
  sign, logdet = _umath_linalg.slogdet(a, signature=signature)


ICC L3 after L2 macro: 0.232
VPC L2 after L2 macro: 0.011

Variance reduction M0 → M2 (the cumulative effect of all controls):
  delta_sigma_u0_sq_pct: +18.2%
  delta_sigma_v0_sq_pct: +58.0%
  delta_sigma_e_sq_pct: +10.1%


/Users/karlalucic/Code/coursework/KUL/2sem/MLA/.venv/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


## 5. Master table M0–M2

In [6]:
tbl = master_table(
    {"M0": res0, "M1": res1, "M2": res2},
    coefs=("genai_i", "gdp_growth", "unemp_rate", "hicp_inflation"),
)
tbl_round = tbl.copy()
for c in tbl_round.select_dtypes("float64").columns:
    tbl_round[c] = tbl_round[c].round(4)
tbl_round

/Users/karlalucic/Code/coursework/KUL/2sem/MLA/.venv/lib/python3.12/site-packages/numpy/linalg/_linalg.py:2349: RuntimeWarning: divide by zero encountered in slogdet
  sign, logdet = _umath_linalg.slogdet(a, signature=signature)
/Users/karlalucic/Code/coursework/KUL/2sem/MLA/.venv/lib/python3.12/site-packages/numpy/linalg/_linalg.py:2349: RuntimeWarning: overflow encountered in slogdet
  sign, logdet = _umath_linalg.slogdet(a, signature=signature)
/Users/karlalucic/Code/coursework/KUL/2sem/MLA/.venv/lib/python3.12/site-packages/numpy/linalg/_linalg.py:2349: RuntimeWarning: invalid value encountered in slogdet
  sign, logdet = _umath_linalg.slogdet(a, signature=signature)


,model,n_obs,n_groups_l3,loglik,aic,bic,sigma_u0_sq,sigma_v0_sq,sigma_e_sq,icc_l3,vpc_l2,genai_i__beta,genai_i__se,gdp_growth__beta,gdp_growth__se,unemp_rate__beta,unemp_rate__se,hicp_inflation__beta,hicp_inflation__se
0,M0,275254,36,-313318.9584,NaN,NaN,0.1910,0.0174,0.5689,0.2457,0.0224,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,M1,187458,36,-205585.5562,NaN,NaN,0.1844,0.0145,0.5225,0.2556,0.0201,0.0882,0.0110,NaN,NaN,NaN,NaN,NaN,NaN
2,M2,165969,30,-180215.0765,NaN,NaN,0.1563,0.0073,0.5113,0.2316,0.0108,0.0923,0.0116,-0.0015,0.0048,-0.0219,0.0053,0.0063,0.006


## 6. Persist results

In [7]:
out_path = INTERIM_DIR / "master_table_m0_m2.parquet"
tbl.to_parquet(out_path, index=False)
print(f"wrote {out_path} ({out_path.stat().st_size/1e3:.1f} KB)")

wrote /Users/karlalucic/Code/coursework/KUL/2sem/MLA/data/interim/master_table_m0_m2.parquet (11.7 KB)


## 7. Day-4 hard checkpoint

From plan §11: *Day 4: confirm L2 VPC ≥ 0.02 (or pre-register narrower contribution claim if not). Flag any 3-level convergence pathology now, not later.*

In [8]:
checks = {
    "L3 ICC in [0.10, 0.30]": 0.10 <= vc0.icc_l3 <= 0.30,
    "L2 VPC ≥ 0.02":          vc0.vpc_l2 >= 0.02,
    "L1 VPC in [0.6, 0.9]":   0.6 <= vc0.vpc_l1 <= 0.9,
    "M1 converged":            res1.converged,
    "M2 converged":            res2.converged,
    "M2 sample > 100k":        len(df_m2) > 100_000,
}
for label, ok in checks.items():
    print(f"  {label}:  {'PASS' if ok else 'FAIL'}")
print()
print("Day-4 checkpoint:", "PASS" if all(checks.values()) else "pending")

  L3 ICC in [0.10, 0.30]:  PASS
  L2 VPC ≥ 0.02:  PASS
  L1 VPC in [0.6, 0.9]:  PASS
  M1 converged:  PASS
  M2 converged:  PASS
  M2 sample > 100k:  PASS

Day-4 checkpoint: PASS
